In [46]:
import configparser
import os

# Initialize the config parser
config = configparser.ConfigParser()
config.read('config.ini')

# Current active profile
user_profile = 'Xuting' 

# Correcting the variable names to match your config.ini keys
# These refer to the 'input =' and 'output =' lines in your file
input_base = config[user_profile]['input']
output_base = config[user_profile]['output']

print(f"Reading data from: {input_base}")
print(f"Saving results to: {output_base}")

Reading data from: /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData
Saving results to: /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData


In [ ]:
import pyarrow.parquet as pq
import polars as pl

# --- Paths to your files ---
# We use os.path.join to combine the base directory with the filename
quotes_file = "data_quotes_2025_06_20.parquet"
trades_file = "data_trades_2025_06_20.parquet"

quotes_path = os.path.join(input_base, quotes_file)
trades_path = os.path.join(input_base, trades_file)

# --- Efficiently count rows without loading data ---
# Now quotes_path and trades_path are correctly defined
quotes_rows = pq.ParquetFile(quotes_path).metadata.num_rows
trades_rows = pq.ParquetFile(trades_path).metadata.num_rows

print(f"Quotes rows: {quotes_rows:,}")
print(f"Trades rows: {trades_rows:,}")

Quotes rows: 1,680,690,895
Trades rows: 101,809,265


In [48]:
import pyarrow.parquet as pq
from datetime import datetime

# ---- quotes file ----
# Construct the path using the base directory from your config file
quotes_file = "data_quotes_2025_06_20.parquet"
quotes_path = os.path.join(input_base, quotes_file)

pf_quotes = pq.ParquetFile(quotes_path)
print("🧾 Quotes schema:")
print(pf_quotes.schema_arrow)     # column names and types only

# ---- trades file ----
# Construct the path using the base directory from your config file
trades_file = "data_trades_2025_06_20.parquet"
trades_path = os.path.join(input_base, trades_file)

pf_trades = pq.ParquetFile(trades_path)
print("\n💹 Trades schema:")
print(pf_trades.schema_arrow)

🧾 Quotes schema:
DATE: date32[day]
TIME_M: time64[us]
EX: string
BID: double
BIDSIZ: int64
ASK: double
ASKSIZ: int64
QU_COND: string
QU_SEQNUM: int64
NATBBO_IND: string
QU_CANCEL: string
QU_SOURCE: string
SYM_ROOT: string
SYM_SUFFIX: string

💹 Trades schema:
DATE: date32[day]
TIME_M: time64[us]
EX: string
SYM_ROOT: string
SYM_SUFFIX: string
TR_SCOND: string
SIZE: int64
PRICE: double
TR_STOP_IND: string
TR_CORR: string
TR_SEQNUM: int64
TR_ID: int64
TR_SOURCE: string
TR_RF: string


In [49]:
#Trades

target_file = os.path.join(input_base, "data_trades_2025_06_20.parquet")
target_output = os.path.join(output_base, "2025_06_20/processed_output_trades/")

#%run polar_try_partitioned.py --FILE_PATH=$target_file --OUTPUT_DIR=$target_output

In [50]:
#Quotes
quotes_input_file = os.path.join(input_base, "data_quotes_2025_06_20.parquet")
quotes_output_dir = os.path.join(output_base, "2025_06_20/processed_output_quotes/")

# Ensure the output directory exists to avoid errors
os.makedirs(quotes_output_dir, exist_ok=True)

#%run polar_try_partitioned.py --FILE_PATH=$quotes_input_file --OUTPUT_DIR=$quotes_output_dir

In [51]:
pl.Config.set_tbl_cols(50) 
pl.Config.set_tbl_width_chars(200)
pl.Config.set_tbl_rows(200) #polars.Config.tbl_rows = 50

polars.config.Config

In [52]:
# Input file remains the same
trades_input = os.path.join(input_base, "data_trades_2025_06_20.parquet")

# Output file with the "_upper" suffix
trades_output_upper = os.path.join(input_base, "data_trades_2025_06_20_upper.parquet")

#%run makesTradesColsUpper.py --INPUT=$trades_input --OUTPUT=$trades_output_upper

pf_trades = pq.ParquetFile(trades_input)
print("\n💹 Trades schema:")
print(pf_trades.schema_arrow)


💹 Trades schema:
DATE: date32[day]
TIME_M: time64[us]
EX: string
SYM_ROOT: string
SYM_SUFFIX: string
TR_SCOND: string
SIZE: int64
PRICE: double
TR_STOP_IND: string
TR_CORR: string
TR_SEQNUM: int64
TR_ID: int64
TR_SOURCE: string
TR_RF: string


In [53]:
from pathlib import Path
import pyarrow.parquet as pq
import os

input_file_upper = os.path.join(input_base, "data_trades_2025_06_20_upper.parquet")
output_dir_clean = os.path.join(output_base, "2025_06_20/processed_output_trades_upper_clean_from_single/")

os.makedirs(output_dir_clean, exist_ok=True)


#%run rewrite2.py --IN_FILE=$input_file_upper --OUT_DIR=$output_dir_clean --BATCH_ROWS=25000


DIR = Path(output_dir_clean)

files = sorted(DIR.glob("*.parquet"))
if not files:
    raise RuntimeError(f"No parquet files found in {DIR}")

# Check the schema of the first generated file to verify results
for f in files[:1]:
    arrow_schema = pq.read_schema(f)
    print(f"File Name: {f.name}")
    print(f"Schema:\n{arrow_schema}")


File Name: chunk_000001.parquet
Schema:
DATE: date32[day]
TIME_M: time64[us]
EX: string
SYM_ROOT: string
SYM_SUFFIX: string
TR_SCOND: string
SIZE: int64
PRICE: double
TR_STOP_IND: string
TR_CORR: string
TR_SEQNUM: int64
TR_ID: int64
TR_SOURCE: string
TR_RF: string


In [54]:
trades_clean_dir = os.path.join(output_base, "2025_06_20/processed_output_trades_upper_clean_from_single")
quotes_processed_dir = os.path.join(output_base, "2025_06_20/processed_output_quotes")

analysis_output_file = os.path.join(output_base, "2025_06_20/taq_analysis_output.txt")

#%run query.py --TRADES_UPPER=$trades_clean_dir --QUOTES_OLD=$quotes_processed_dir --OUTPUT_FILE=$analysis_output_file


In [55]:
import os
import polars as pl


quotes_path = os.path.join(input_base, "data_quotes_2025_06_20.parquet")

q = pl.scan_parquet(quotes_path).filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()

len(q)

22604986

In [56]:
QUOTES_DIR = os.path.join(output_base, "2025_06_20/processed_output_quotes/")
q = pl.scan_parquet(f"{QUOTES_DIR}/*.parquet").filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()
len(q)

22604986

In [57]:
QUOTES_DIR = os.path.join(output_base, "2025_06_20/processed_output_quotes_top50/")
q = pl.scan_parquet(f"{QUOTES_DIR}/*.parquet").filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()
len(q)

22604986

In [58]:
trades_path = os.path.join(input_base, "data_trades_2025_06_20.parquet")

# Process data
t = pl.scan_parquet(trades_path).filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()

len(t)

825571

In [59]:
trades_upper_path = os.path.join(input_base, "data_trades_2025_06_20_upper.parquet")

t = pl.scan_parquet(trades_upper_path).filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()

len(t)

825571

In [60]:
TRADES_DIR = os.path.join(output_base, "2025_06_20/processed_output_trades/")

q = pl.scan_parquet(f"{TRADES_DIR}/*.parquet").filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()

len(q)

825571

In [61]:
import polars as pl

schema = {
    "DATE": pl.Date,
    "TIME_M": pl.Time,
    "EX": pl.Utf8,
    "SYM_ROOT": pl.Utf8,
    "SYM_SUFFIX": pl.Utf8,
    "TR_SCOND": pl.Utf8,
    "SIZE": pl.Int64,
    "PRICE": pl.Float64,
    "TR_STOP_IND": pl.Utf8,
    "TR_CORR": pl.Utf8,
    "TR_SEQNUM": pl.Int64,
    "TR_ID": pl.Int64,
    "TR_SOURCE": pl.Utf8,
    "TR_RF": pl.Utf8,
}

TRADES = os.path.join(input_base, "data_trades_2025_06_20_upper.parquet")

t = pl.scan_parquet(TRADES, schema=schema).filter(
    (pl.col("SYM_ROOT") == "SPY") & 
    (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()
print(len(t))

t = pl.scan_parquet(TRADES, schema=schema).collect()
print(len(t))

825571
101809265


In [62]:
import polars as pl

schema = {
    "DATE": pl.Date,
    "TIME_M": pl.Time,
    "EX": pl.Utf8,
    "SYM_ROOT": pl.Utf8,
    "SYM_SUFFIX": pl.Utf8,
    "TR_SCOND": pl.Utf8,
    "SIZE": pl.Int64,
    "PRICE": pl.Float64,
    "TR_STOP_IND": pl.Utf8,
    "TR_CORR": pl.Utf8,
    "TR_SEQNUM": pl.Int64,
    "TR_ID": pl.Int64,
    "TR_SOURCE": pl.Utf8,
    "TR_RF": pl.Utf8,
}



# TRADES_CLEAN = os.path.join(output_base, "2025_06_20/processed_output_trades_upper_clean/")
# t_clean_spy = pl.scan_parquet(TRADES_CLEAN, schema=schema).filter(
#   (pl.col("SYM_ROOT") == "SPY") & (pl.col("SYM_SUFFIX").fill_null("") == "")
# ).collect()
# print(f"Clean SPY: {len(t_clean_spy)}")

TRADES_SINGLE = os.path.join(output_base, "2025_06_20/processed_output_trades_upper_clean_from_single/")


t_single_spy = pl.scan_parquet(TRADES_SINGLE).filter(
    (pl.col("SYM_ROOT") == "SPY") & (pl.col("SYM_SUFFIX").fill_null("") == "")
).collect()
print(f"Single SPY: {len(t_single_spy)}")


t_single_all = pl.scan_parquet(TRADES_SINGLE).collect()
print(f"Single Total: {len(t_single_all)}")

Single SPY: 825571
Single Total: 101809265


In [63]:
# %run offending_datatypes.py --TRADES_DIR=/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/processed_output_trades_upper_clean_from_single

In [64]:
# %run query.py  --TRADES_UPPER="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/processed_output_trades_upper_clean_from_single/*.parquet" --QUOTES_OLD="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/processed_output_quotes/*.parquet" --OUTPUT_FILE="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/taq_analysis_output.txt"
# trades_clean = os.path.join(output_base, "2025_06_20/processed_output_trades_upper_clean_from_single/*.parquet")
# quotes_old = os.path.join(output_base, "2025_06_20/processed_output_quotes/*.parquet")
# output_txt = os.path.join(output_base, "2025_06_20/taq_analysis_output.txt")

# %run query.py --TRADES_UPPER=$trades_clean --QUOTES_OLD=$quotes_old --OUTPUT_FILE=$output_txt

In [65]:
# %run stats.py --TRADES_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/processed_output_trades_upper_clean_from_single/" --QUOTES_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/processed_output_quotes/" --OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/stats_out_simple/"
# t_dir = os.path.join(output_base, "2025_06_20/processed_output_trades_upper_clean_from_single/")
# q_dir = os.path.join(output_base, "2025_06_20/processed_output_quotes/")
# o_dir = os.path.join(output_base, "2025_06_20/stats_out_simple/")

# %run stats.py --TRADES_DIR=$t_dir --QUOTES_DIR=$q_dir --OUT_DIR=$o_dir

In [66]:
OUT_DIR = os.path.join(output_base, "2025_06_20/stats_out_simple")
TRADES_OUT_CSV = os.path.join(OUT_DIR, "top50_trades_by_volume.csv")
QUOTES_OUT_CSV = os.path.join(OUT_DIR, "top50_quotes_by_rows.csv")

print(f"Reading {TRADES_OUT_CSV} and {QUOTES_OUT_CSV} …")
trades_df = pl.read_csv(TRADES_OUT_CSV)
quotes_df = pl.read_csv(QUOTES_OUT_CSV)

print("\n🔥 Top Trades (Sorted by TRADES_ROWS):")
print(trades_df.sort("TRADES_ROWS", descending=True))

print("\n📊 Top Quotes (Sorted by QUOTES_ROWS):")
print(quotes_df.sort("QUOTES_ROWS", descending=True))

Reading /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/stats_out_simple/top50_trades_by_volume.csv and /Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/stats_out_simple/top50_quotes_by_rows.csv …

🔥 Top Trades (Sorted by TRADES_ROWS):
shape: (50, 4)
┌──────────┬────────────┬─────────────┬────────────┐
│ SYM_ROOT ┆ SYM_SUFFIX ┆ TRADES_ROWS ┆ TRADES_VOL │
│ ---      ┆ ---        ┆ ---         ┆ ---        │
│ str      ┆ str        ┆ i64         ┆ i64        │
╞══════════╪════════════╪═════════════╪════════════╡
│ CRCL     ┆            ┆ 1493428     ┆ 102332349  │
│ NVDA     ┆            ┆ 1438654     ┆ 299211330  │
│ TSLA     ┆            ┆ 1326290     ┆ 117276951  │
│ SPY      ┆            ┆ 825571      ┆ 107271149  │
│ GOOG     ┆ L          ┆ 782179      ┆ 92761295   │
│ AMD      ┆            ┆ 693964      ┆ 88889788   │
│ AAPL     ┆            ┆ 680935      ┆ 127847384  │
│ PLTR     ┆            ┆ 629950      ┆ 110668010  │
│ AMZN     ┆            ┆ 5

In [67]:
# %run top50_trades_corresponding_quotes.py --OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/stats_out_simple/" --TRADES_TOP50_CSV=top50_trades_by_volume.csv --TRADES_SRC_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/processed_output_trades_upper_clean_from_single/" --TRADES_OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/processed_output_trades_upper_top50/" --QUOTES_SRC_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/processed_output_quotes/" --QUOTES_OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/processed_output_quotes_top50/" --MERGED_OUT_CSV="merged_top50_sorted.csv"
# import os


# stats_dir = os.path.join(output_base, "2025_06_20/stats_out_simple/")
# top50_csv = "top50_trades_by_volume.csv"

# trades_src = os.path.join(output_base, "2025_06_20/processed_output_trades_upper_clean_from_single/")
# trades_out = os.path.join(output_base, "2025_06_20/processed_output_trades_upper_top50/")

# quotes_src = os.path.join(output_base, "2025_06_20/processed_output_quotes/")
# quotes_out = os.path.join(output_base, "2025_06_20/processed_output_quotes_top50/")

# os.makedirs(trades_out, exist_ok=True)
# os.makedirs(quotes_out, exist_ok=True)

# --- 2. Run the Top 50 Processing Script ---
# %run top50_trades_corresponding_quotes.py \
#     --OUT_DIR=$stats_dir \
#     --TRADES_TOP50_CSV=$top50_csv \
#     --TRADES_SRC_DIR=$trades_src \
#     --TRADES_OUT_DIR=$trades_out \
#     --QUOTES_SRC_DIR=$quotes_src \
#     --QUOTES_OUT_DIR=$quotes_out \
#     --MERGED_OUT_CSV="merged_top50_sorted.csv"

In [68]:
# %run top50_trades_corresponding_quotes_persist.py --TRADES_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/processed_output_trades_upper_top50/" --QUOTES_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/processed_output_quotes_top50/" --STATS_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/stats_out_simple/" --TOP50_CSV="top50_trades_by_volume.csv" --OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/merged_output_top50/" --OUT_FILE="merged_all.parquet" --CHUNK_SIZE=25000

# t_dir = os.path.join(output_base, "2025_06_20/processed_output_trades_upper_top50/")
# q_dir = os.path.join(output_base, "2025_06_20/processed_output_quotes_top50/")
# s_dir = os.path.join(output_base, "2025_06_20/stats_out_simple/")
# out_dir = os.path.join(output_base, "2025_06_20/merged_output_top50/")

# %run top50_trades_corresponding_quotes_persist.py \
#    --TRADES_DIR=$t_dir \
#    --QUOTES_DIR=$q_dir \
#    --STATS_DIR=$s_dir \
#    --TOP50_CSV="top50_trades_by_volume.csv" \
#    --OUT_DIR=$out_dir \
#    --OUT_FILE="merged_all.parquet" \
#    --CHUNK_SIZE=25000

In [69]:
%run ms.py --MERGED_FILE="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/merged_output_top50/merged_all.parquet" --OUT_DIR="/Users/alessiaaaa/Desktop/WithAmitG/WithAmitG/TAQData/2025_06_20/ms_out/" --TICKERS=AAPL,SPY,TSLA


merged_file = os.path.join(output_base, "2025_06_20/merged_output_top50/merged_all.parquet")
ms_out_dir = os.path.join(output_base, "2025_06_20/ms_out/")

%run ms.py --MERGED_FILE=$merged_file --OUT_DIR=$ms_out_dir --TICKERS=AAPL,SPY,TSLA

Null TR_SEQNUM : 0
Null QU_SEQNUM : 268376
Zero BID and ASK : 35721

🔎 Running microstructure analysis for: AAPL
shape: (1, 19)
┌────────┬────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┐
│ TICKER ┆ N_ROWS ┆ TOTAL_SIZE ┆ VWAP       ┆ AVG_SPREAD ┆ AVG_DEPTH_ ┆ IS_MEAN_BP ┆ IS_MEDIAN_ ┆ IS_MIN_BPS ┆ IS_MAX_BPS ┆ MI_MEAN_BP ┆ MI_MEDIAN_ ┆ MI_MIN_BPS ┆ MI_MAX_BPS ┆ MR_MEAN_BP ┆ MR_MEDIAN_ ┆ MR_MIN_BPS ┆ MR_MAX_BPS ┆ FWD_TRADES │
│ ---    ┆ ---    ┆ ---        ┆ ---        ┆ _BPS       ┆ TOB        ┆ S          ┆ BPS        ┆ ---        ┆ ---        ┆ S          ┆ BPS        ┆ ---        ┆ ---        ┆ S          ┆ BPS        ┆ ---        ┆ ---        ┆ _FOR_MR    │
│ str    ┆ u32    ┆ i64        ┆ f64        ┆ ---        ┆ ---        ┆ ---        ┆ ---        ┆ f64        ┆ f64        ┆ ---        ┆ ---        ┆

In [70]:
summary_df

TICKER,N_ROWS,TOTAL_SIZE,VWAP,AVG_SPREAD_BPS,AVG_DEPTH_TOB,IS_MEAN_BPS,IS_MEDIAN_BPS,IS_MIN_BPS,IS_MAX_BPS,MI_MEAN_BPS,MI_MEDIAN_BPS,MI_MIN_BPS,MI_MAX_BPS,MR_MEAN_BPS,MR_MEDIAN_BPS,MR_MIN_BPS,MR_MAX_BPS,FWD_TRADES_FOR_MR
str,u32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32
"""AAPL""",671773,123558690,199.868856,4874.696363,6.373384,7.112085,0.395377,0.0,2540.595185,0.0,0.0,-0.0,0.0,-5.576774,-0.251781,-2503.633032,2019.061066,10
"""SPY""",820984,100216919,595.425057,1064.209778,7.783044,0.555265,0.084063,0.0,3009.014916,0.0,0.0,-0.0,0.0,-0.450153,-0.083945,-3008.250014,937.557794,10
"""TSLA""",1315540,116595642,323.037077,15365.727382,5.691688,10.546481,1.099816,0.0,807.017544,0.0,0.0,-0.0,0.0,-8.79226,-0.775542,-877.297429,874.030889,10
